In [1]:
# %pip install python-dotenv
# %uv add dspy

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))


### check aicodetools library

In [3]:
import time
import threading
import tiktoken
from collections import deque
import dspy
from dspy.utils.callback import BaseCallback


class SlidingWindowLimiter:
    """Rate limiter that enforces both request and token limits per rolling minute."""

    _instance = None
    _lock = threading.Lock()

    def __new__(cls, max_requests_per_min=1000, max_tokens_per_min=2_000_000):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance.max_requests = max_requests_per_min
            cls._instance.max_tokens = max_tokens_per_min
            cls._instance.requests = deque()  # [(timestamp, tokens)]
            cls._instance._lock = threading.Lock()
            cls._instance.encoder = tiktoken.get_encoding("cl100k_base")
        return cls._instance

    def _cleanup(self, now):
        """Remove entries older than 60s."""
        while self.requests and now - self.requests[0][0] > 60:
            self.requests.popleft()

    def _count(self):
        """Total requests & tokens in current 60s window."""
        total_tokens = sum(t for _, t in self.requests)
        return len(self.requests), total_tokens

    def acquire(self, tokens_used=0):
        """Wait until request fits in sliding 60s window."""
        with self._lock:
            while True:
                now = time.time()
                self._cleanup(now)
                req_count, token_count = self._count()

                # Can fit in current 60s window?
                if (req_count < self.max_requests and
                        token_count + tokens_used <= self.max_tokens):
                    # Record the new request
                    self.requests.append((now, tokens_used))
                    break  # proceed

                # Otherwise, figure out when we can retry
                oldest_time = self.requests[0][0]
                sleep_time = max(0.01, 60 - (now - oldest_time))
                print(f"⚠️ Throttling: sleeping {sleep_time:.2f}s (req={req_count}, tokens={token_count})")
                time.sleep(sleep_time)


class DelayAndLogCallback(BaseCallback):
    """DSPy callback using sliding window limiter."""

    def __init__(self):
        self.limiter = SlidingWindowLimiter()

    def _estimate_tokens(self, messages=None, prompt=None):
        """Estimate token usage using tiktoken."""
        text = ""
        if prompt:
            text = str(prompt)
        elif messages:
            # concatenate all message contents
            text = " ".join(m.get("content", "") for m in messages)
        return len(self.limiter.encoder.encode(text))

    def on_lm_start(self, *args, **kwargs):
        inputs = kwargs.get("inputs") or {}
        prompt = inputs.get("prompt")
        messages = inputs.get("messages")
        tokens_used = self._estimate_tokens(messages=messages, prompt=prompt)
        self.limiter.acquire(tokens_used=tokens_used)

    def on_lm_end(self, *args, **kwargs):
        pass


In [4]:

# from aicodetools import ClientManager 

# code_tool_manager = ClientManager(
#                 "super-bench:latest", base_log_dir="runs/super/"
#             )

# code_tool_client = code_tool_manager.get_client('initial')

In [5]:
import os
# os.environ['OPENAI_API_KEY'/] = input()
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

True

In [6]:
import dspy
lm = dspy.LM("azure/gpt-4.1", temperature=1.0, num_retries=30, callbacks=[DelayAndLogCallback()])
tlm = dspy.LM("azure/gpt-4.1",temperature=1.0)
dspy.configure(lm=lm)

In [7]:
# print(lm("Say this is a test!") ) # => ['This is a test!']
print(lm(messages=[{"role": "user", "content": "Say this is a test!"}]))  # => ['This is a test!']
print(tlm(messages=[{"role": "user", "content": "t : Say this is a test!"}]))  # => ['This is a test!']

['This is a test!']
['This is a test!']


## Load the benchmark and view one example from the benchmark

In [8]:

from gepa_artifact.benchmarks.super_bench.super_utils import FinishResponse
from gepa_artifact.benchmarks.super_bench import benchmark as sb_metas

Available Tools for 91c6fa10-abc6-454c-974d-c89d520c4efe: on runtime aicodetools-91c6fa10-abc6-454c-974d-c89d520c4efe-e4b314e9 4


In [9]:
bench = sb_metas[0].benchmark()

In [10]:
len(bench.train_set), len(bench.val_set), len(bench.test_set)

(9, 9, 27)

In [11]:
import pprint
pprint.pprint(bench.train_set[0])

Example({'instance_id': 'pie-perf', 'github_repo': 'https://github.com/madaan/pie-perf', 'git_commit': 'ee1989b66756470622e3b89c4aa031f083f57ef9', 'query': 'Evaluate the generations of my code improving model which are provided in https://drive.google.com/file/d/1izs1iF5cd_NAZsOaZvrrQF3NAsoP8lHf/view?usp=sharing (v1 vs v0). Once evaluated, report the result problem_id and input_acc for each problem of the dataset, as a json list of dictionaries structured as follows: [{"problem_id": "", "input_acc": 0.0}] (replace "" and 0.0 with the actual values).\n\nAdditional instructions:\n1. Set "num_trials": 2 in the evaluation configuration file to reduce computation time.\n2. Load only the first 10 rows of the dataset.\n\nGit repository: https://github.com/madaan/pie-perf', 'query_components': {'e2e_task': 'Evaluate the generations of my code improving model which are provided in https://drive.google.com/file/d/1izs1iF5cd_NAZsOaZvrrQF3NAsoP8lHf/view?usp=sharing (v1 vs v0).', 'scenario_task': '

## Load the program and display the program
The program is a 3-module system, each of which handles the urgency, sentiment and categories classification respectively

In [12]:
program = sb_metas[0].program[0]
program

react.react = Predict(StringSignature(query, github_repo, git_commit, trajectory -> next_thought, next_tool_name, next_tool_args
    instructions='Solve the question and provide the answer in the correct format.\n\nYou are an Agent. In each episode, you will be given the fields `query`, `github_repo`, `git_commit` as input. And you can see your past trajectory so far.\nYour goal is to use one or more of the supplied tools to collect any necessary information for producing `result`.\n\nTo do this, you will interleave next_thought, next_tool_name, and next_tool_args in each turn, and also when finishing the task.\nAfter each tool call, you receive a resulting observation, which gets appended to your trajectory.\n\nWhen writing next_thought, you may reason about the current situation and plan for future steps.\nWhen selecting the next_tool_name and its next_tool_args, the tool must be one of:\n\n(1) read_file, whose description is <desc>          Read file with optional line range and/or 

### Make Sure docker is installed and running

## Define an evaluator and evaluate the base program

In [13]:
import dspy
evaluate = dspy.Evaluate(
    devset=bench.test_set,
    metric=sb_metas[0].metric,
    num_threads=9,
    display_table=True,
    display_progress=True,
    max_errors=100 * len(bench.test_set),
    provide_traceback = True,
    failure_score=0,
    save_as_json=''
)

In [ ]:
evaluate(program)

  0%|          | 0/27 [00:00<?, ?it/s]Available Tools for g-transformer: on runtime aicodetools-g-transformer-4f1fe7a5 4
Available Tools for spa: on runtime aicodetools-spa-89453ed6 4
Available Tools for mezo: on runtime aicodetools-mezo-34e83801 4
Available Tools for mode-connectivity-plm: on runtime aicodetools-mode-connectivity-plm-ba42bbb7 4
Available Tools for mbib: on runtime aicodetools-mbib-b74392b0 4
Available Tools for unsupervisedhierarchicalsymbolicregression: on runtime aicodetools-unsupervisedhierarchicalsymbolicregression-c81597b7 4
Available Tools for conv_graph: on runtime aicodetools-conv_graph-c5a28dfa 4
Available Tools for pira: on runtime aicodetools-pira-4cfac969 4
Available Tools for pet: on runtime aicodetools-pet-2a649d12 4
Cleaned up Tools g-transformer : True  True
success=False structured_output={'Sentence-level BLEU': 'None', 'Document-level BLEU': 'None'} reasoning='The g-transformer repository is not present in the workspace; directory listings show no pr

2025/10/20 12:42:28 WARNING dspy.predict.react: Ending the trajectory: Agent failed to select a valid tool: 
Traceback (most recent call last):
  File "/mnt/c/Users/2825425/work/gepa-research/gepa-modified/gepa_artifact/utils/dspy/dspy/adapters/base.py", line 33, in _call_post_process
    value = self.parse(signature, output)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/2825425/work/gepa-research/gepa-modified/gepa_artifact/utils/dspy/dspy/utils/callback.py", line 326, in sync_wrapper
    return fn(instance, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/2825425/work/gepa-research/gepa-modified/gepa_artifact/utils/dspy/dspy/adapters/chat_adapter.py", line 175, in parse
    raise ValueError(f"Expected {signature.output_fields.keys()} but got {fields.keys()}", len(fields.keys()), len(signature.output_fields.keys()))
ValueError: ("Expected dict_keys(['next_thought', 'next_tool_name', 'next_tool_args']) but got dict_keys(['next_thought'])",

Cleaned up Tools memorizing-transformers-pytorch : True  True
success=False structured_output={} reasoning='My solution was blocked by repeated failures in writing the modified train.py file. The errors were due to passing non-string (fragmented/dict) arguments instead of the full, joined script source. Although the correct file updates and process are clear, I was unable to complete the experiment and report the validation loss.' summary='Cloned repo, found the correct train.py, read and planned edits (seed, batches, batch-size, segments, validation interval) but was blocked by persistent formatting errors when writing the modified file. No training run or result could be completed.'
I encountered persistent failures in updating train.py due to incorrectly formatted content passed to the file writing tool (write_file). The content argument must be a single valid string of the entire modified Python file, not fragments or dict keys. My initial attempts mistakenly included dictionary fr

Cleaned up Tools dpt : True  True
success=False structured_output=None reasoning='All required steps for setting up training and metric reporting for decomposed prompt tuning were planned, including code changes for custom dataset integration. However, repeated workspace, filesystem, and network errors made the codebase and dataset inaccessible. All critical steps after setup failed due to environment/network issues and the inability to access or modify files.' summary='Unable to complete the integration and training due to persistent workspace and network failures. Repo and file access were lost, preventing the experiment and metric reporting. If the environment is restored, the process should be retried from repo and data setup onward.'
The main steps for applying decomposed prompt tuning to fine-tune t5-small on the custom sentence pair classification dataset were planned as follows:
1. Clone the required GitHub repo and check out the desired commit.
2. Download and inspect the Goog

Cleaned up Tools upet : True  True
success=False structured_output={'eval_accuracy': None} reasoning='The correct command and code modifications were identified and made, but repeated environment and filesystem issues (file/directory disappearances) after each step prevented a successful run and access to the required files. As a result, it was not possible to complete the training and evaluation, nor retrieve the eval accuracy.' summary='I cloned the repo, extracted the correct training command, and modified code imports for compatibility with recent Hugging Face libraries. After installing dependencies and attempting to run the training, file/directory disappearances blocked progression. The workspace instability prevented me from running the experiment and reporting eval accuracy.'
My objective was to train a roberta-base model on the RTE dataset using UPET, running only 1 epoch with 5 examples per label and seed=42, and then report the evaluation accuracy as a JSON object. 

Here a

...

Cleaned up Tools pet : True  True
success=False structured_output={'acc-p0': 0.0, 'acc-all-p': 0.0} reasoning='The necessary code files and dataset logic could not be accessed due to persistent server connection issues, so it was impossible to configure or run the requested PET training/evaluation experiment for MNLI or report final results as specified.' summary='I attempted to find the entry script and inspect code/data logic to control number of examples and experiment setup, but hit repeated connection errors on all critical file accesses, blocking progress.'
I attempted to fulfill the request by first locating the correct entry script for training and evaluating PET models on the MNLI dataset with pattern IDs 0 and 1 using bert-base-uncased. After determining the main script is likely `run.py` under the UPET2 directory (since `cli.py` was not found), I investigated the logic for limiting the number of examples, passing pattern IDs, model selection, and epoch count. However, I enco

2025/10/20 12:58:24 INFO dspy.evaluate.evaluate: Average Metric: 2.541666666666667 / 27 (9.4%)


,instance_id,github_repo,git_commit,query,query_components,answer,landmarks,trajectory,reasoning,result,super_score
0,g-transformer,https://github.com/baoguangsheng/g-transformer,dcc7695ceb0ecc3250e1c28215e9ddcd22700b39,Use the https://github.com/baoguangsheng/g-transformer repository ...,{'e2e_task': 'Use the https://github.com/baoguangsheng/g-transform...,"{""Sentence-level BLEU"": 0.0, ""Document-level BLEU"": 0.01}",['INFO\\] Building segmented data' 'INFO \\| fairseq_cli.preproces...,{'thought_0': 'I need to determine how fine-tuning is performed in...,The repository (https://github.com/baoguangsheng/g-transformer) is...,"success=False structured_output={'Sentence-level BLEU': 'None', 'D...","✔️ [Prediction(\n score=0.0,\n score_dict={'submitted': 1, '..."
1,spa,https://github.com/OceannTwT/SPA,a8bb190a182c076f80d90ac59921abd1df29b0ae,"Train the SPA model on alpaca_data_en_52k (from the repo), startin...",{'e2e_task': 'Train the SPA model on alpaca_data_en_52k (from the ...,"{""training_loss"": 4.97342586517334}","[Generating train split: \d+ examples, >> \*\*\*\*\* Running train...","{'thought_0': 'To follow the instructions, I first need to inspect...",The task was to train the SPA model on `alpaca_data_en_52k` using ...,success=False structured_output={'training_loss': None} reasoning=...,"✔️ [Prediction(\n score=0.0,\n score_dict={'submitted': 1, '..."
2,mezo,https://github.com/princeton-nlp/MeZO,552cb1b710767f9a6e1dc8f9645d7640376f9941,"Train using the ""MeZO"" method (no prefix-tuning / lora) on the RTE...","{'e2e_task': 'Train using the ""MeZO"" method (no prefix-tuning / lo...","{""accuracy"": 0.8, ""dev_accuracy"": 0.4}",['- INFO - Sample train set \\d+/\\d+' '- INFO - \\*\\*\\*\\*\\* R...,"{'thought_0': 'To train on RTE with MeZO\'s method, we need to loc...",The experimental MeZO training for RTE requires the scripts and da...,success=False structured_output=None reasoning='The necessary scri...,"✔️ [Prediction(\n score=0.0,\n score_dict={'submitted': 0, '..."
3,mode-connectivity-plm,https://github.com/thunlp/mode-connectivity-plm,f45efd03fabdb23777c3ddcb35612cb1f7b1960c,"Fine-tune roberta-base on a single-task, ""rotten tomatoes"" which i...","{'e2e_task': 'Fine-tune roberta-base on a single-task, ""rotten tom...","{""eval_loss"": 0.0368}","[Running tokenizer on dataset:, \*\*\*\*\* train metrics \*\*\*\*\...","{'thought_0': ""To use the hyperparameters for fine-tuning, I need ...","The objective was to fine-tune ""roberta-base"" on the ""rotten_tomat...",success=False structured_output=None reasoning='Although the mnli....,"✔️ [Prediction(\n score=0.0,\n score_dict={'submitted': 0, '..."
4,mbib,https://github.com/Media-Bias-Group/MBIB,b9a887ffd461fa462e89835fc27b36e370091954,"Train a bart-base model on the ""linguistic-bias"" task using the ba...","{'e2e_task': 'Train a bart-base model on the ""linguistic-bias"" tas...","{""average_weighted_f1"": 0.44272727272727275}","[Training Initialized for fold \d+, The current dev loss: tensor\(...","{'thought_0': ""To proceed, I need to locate the generated `linguis...",The requested experiment cannot be completed as-is due to missing ...,success=False structured_output={'average_weighted_f1': None} reas...,"✔️ [Prediction(\n score=0.0,\n score_dict={'submitted': 1, '..."
5,unsupervisedhierarchicalsymbolicregression,https://github.com/SiyuLou/UnsupervisedHierarchicalSymbolicRegression,9fb1ba66ee1006f14c15f1b5b2390c8d0fa24fd6,Train a hierarchical symbolic regression model from this repo usin...,{'e2e_task': 'Train a hierarchical symbolic regression model from ...,"{""Rf_final"": {""R_square"": -0.25985264778137207, ""MSE"": 0.148261207...","[the size of train dataset is \d+, validation_dataset is \d+, test...","{'thought_0': ""To begin, I need to locate the training script that...","To solve this task, I cloned the UnsupervisedHierarchicalSymbolicR...",success=True structured_output={'Rf_final': {'R_square': -0.259852...,"✔️ [Prediction(\n score=1.0,\n score_dict={'subm

9.41

2025/10/20 14:19:05 ERROR dspy.utils.parallelizer: Error for Example({'instance_id': 'pet', 'github_repo': 'https://github.com/timoschick/pet', 'git_commit': '21d32de975a911bfa0261827c9bd23dc4f0e4aa2', 'query': 'Train and evaluate a PET model on the MNLI dataset using the `cli.py` script with PATTERN_IDS 0 and 1, with bert-base-uncased as the base model. Report results.\nMNLI can be found in the glue dataset: https://drive.google.com/file/d/1s5yJzvU0y9KkACjch9RDiBwObpNXt-Rz/view?usp=drive_link Report final acc-p0 and acc-all-p as a json structured as follows: {"acc-p0": 0.0, "acc-all-p": 0.0} (replace 0.0 with the actual values).\n\nAdditional instructions:\n1. Only load the first 10 examples of each (training/validation/test) set in the dataset.\n2. Run for only 1 epoch.\n\nGit repository: https://github.com/timoschick/pet', 'query_components': {'e2e_task': 'Train and evaluate a PET model on the MNLI dataset using the `cli.py` script with PATTERN_IDS 0 and 1, with bert-base-uncased as

## Load the GEPA Optimizer

In [ ]:
# Import GEPA and define the optimizer
from gepa_artifact.gepa.gepa import GEPA
from gepa_artifact.utils.capture_stream_logger import Logger

import time

runs_dir = os.path.join(os.getcwd(), "runs", time.strftime("%Y-%m-%d_%H-%M-%S"))
os.makedirs(runs_dir, exist_ok=True)

gepa_logger = Logger(os.path.join(runs_dir, "run_log.txt"))

if sb_metas[0].feedback_fn_maps is None or sb_metas[0].feedback_fn_maps[0] is None:
    def feedback_func(predictor_output, predictor_inputs, module_inputs, module_outputs, captured_trace):
        pred = sb_metas[0].metric_with_feedback(module_inputs, module_outputs, None)
        return {
            "feedback_score": pred.score,
            "feedback_text": pred.feedback,
        }

    feedback_fn_map = {k:feedback_func for k, v in program.named_predictors()}
else:
    feedback_fn_map = sb_metas[0].feedback_fn_maps[0]

optimizer = GEPA(
    named_predictor_to_feedback_fn_map=feedback_fn_map,
    knowledgebase_qe=None,
    metric=sb_metas[0].metric,
    run_linearized_gepa=False,
    use_merge=True, 
    teacher_lm = tlm,
    set_for_merge_minibatch='val', 
    track_scores_on='val',
    num_iters=9,
    max_metric_calls=150,
    run_dir=runs_dir,
    logger=gepa_logger,
    num_threads=9)

## Optimize the program with GEPA

In [16]:
sb_metas[0].program[0].get_lm()

In [ ]:
optimized_program = optimizer.compile(
    sb_metas[0].program[0],
    trainset=bench.train_set,
    valset=bench.val_set,
)

2025/10/20 09:06:50 ERROR dspy.utils.parallelizer: Error for Example({'instance_id': 'multi3woz', 'github_repo': 'https://github.com/cambridgeltl/multi3woz', 'git_commit': 'c65e80fe120704125255de5cf582a51ebaa285cd', 'query': 'Train and evaluate a slot labelling model on the French language data with xlm-roberta-base as the base model. Report the loss, accuracy and f1 on both the validation and test sets as a json structured as follows: {"validation": {"loss": 0.0, "f1": 0.0, "accuracy": 0.0}, "test": {"loss": 0.0, "f1": 0.0, "accuracy": 0.0}} (replace 0.0 with the actual values).\n\nAdditional instructions:\n1. Train for 1 epoch.\n2. Use only the first 10 entries each of train, dev, and test sets during training.3. Use the following hyperparameters: task = labelling, language = French, seed = 1, batch_size = 64, training_epoch = 1, process_mode = user, context_window = 3, learning_rate = 2e-5, weight_decay = 0.1, max_context_char_length = 150\n\nGit repository: https://github.com/cambr

In [ ]:
optimizer.gepa_state.save(runs_dir)



In [19]:
def idxmax(lst):
    """Return the index of the maximum value in a list."""
    max_val = max(lst)
    return lst.index(max_val)

In [20]:
gepa_state = optimizer.gepa_state
best_prog_idx = idxmax(gepa_state.per_program_tracked_scores)
best_prog = gepa_state.program_candidates[best_prog_idx]

In [22]:
optimized_program=best_prog

## Now, let's evaluate the optimized program

In [23]:
evaluate(optimized_program)

  0%|          | 0/27 [00:00<?, ?it/s]Available Tools for g-transformer: on runtime aicodetools-g-transformer-018464ad 4
Available Tools for spa: on runtime aicodetools-spa-1c293ae3 4
Available Tools for mezo: on runtime aicodetools-mezo-723a0405 4
Available Tools for mode-connectivity-plm: on runtime aicodetools-mode-connectivity-plm-770f1895 4
Available Tools for mbib: on runtime aicodetools-mbib-c5fe66d3 4
Available Tools for unsupervisedhierarchicalsymbolicregression: on runtime aicodetools-unsupervisedhierarchicalsymbolicregression-0daa9628 4
Available Tools for conv_graph: on runtime aicodetools-conv_graph-8e5204e7 4
Available Tools for pira: on runtime aicodetools-pira-e53a9588 4
Available Tools for pet: on runtime aicodetools-pet-ecd618dd 4
Cleaned up Tools g-transformer : True  True
success=False structured_output={'Sentence-level BLEU': 0.0, 'Document-level BLEU': 0.0} reasoning="The specified repository (https://github.com/baoguangsheng/g-transformer) is absent in the workin

2025/10/20 08:29:49 WARNING dspy.predict.react: Ending the trajectory: Agent failed to select a valid tool: 
Traceback (most recent call last):
  File "/mnt/c/Users/2825425/work/gepa-research/gepa-modified/gepa_artifact/utils/dspy/dspy/adapters/chat_adapter.py", line 169, in parse
    fields[k] = parse_value(v, signature.output_fields[k].annotation)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/2825425/work/gepa-research/gepa-modified/gepa_artifact/utils/dspy/dspy/adapters/utils.py", line 173, in parse_value
    return TypeAdapter(annotation).validate_python(candidate)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/2825425/work/gepa-research/gepa-modified/.venv/lib/python3.12/site-packages/pydantic/type_adapter.py", line 441, in validate_python
    return self.validator.validate_python(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
pydantic_core._pydantic_core.ValidationError: 1 validation error for dict[str,

Cleaned up Tools mezo : True  True
success=False structured_output={'accuracy': 0.0, 'dev_accuracy': 0.0} reasoning='Step 1 (Repository Preparation) failed: The workspace directory is empty, with no files or subdirectories. The MeZO repository (https://github.com/princeton-nlp/MeZO at commit 552cb1b710767f9a6e1dc8f9645d7640376f9941) was not cloned, so all subsequent protocol steps (code inspection, dataset loading, script execution) are blocked. As per protocol, no experiment can be run until the repository and required scripts are present. This output reflects early termination with placeholders, per strict instructions.' summary='Experiment was terminated at Step 1 because the MeZO repository was missing from the workspace. No files or scripts were available to inspect or execute. To proceed, the repository must be cloned at the specified commit.'
Stepwise protocol review:

Step 1: Repository Preparation  
- Checked the current working directory (/workspace) and listed contents with 

2025/10/20 08:41:27 INFO dspy.evaluate.evaluate: Average Metric: 2.458333333333333 / 27 (9.1%)


,instance_id,github_repo,git_commit,query,query_components,answer,landmarks,trajectory,reasoning,result,super_score
0,g-transformer,https://github.com/baoguangsheng/g-transformer,dcc7695ceb0ecc3250e1c28215e9ddcd22700b39,Use the https://github.com/baoguangsheng/g-transformer repository ...,{'e2e_task': 'Use the https://github.com/baoguangsheng/g-transform...,"{""Sentence-level BLEU"": 0.0, ""Document-level BLEU"": 0.01}",['INFO\\] Building segmented data' 'INFO \\| fairseq_cli.preproces...,"{'thought_0': ""Before configuring or running the experiment, I nee...",Stepwise reasoning: - Step 1 (Repository Preparation): Attempted t...,"success=False structured_output={'Sentence-level BLEU': 0.0, 'Docu...","✔️ [Prediction(\n score=0.25,\n score_dict={'submitted': 1, ..."
1,spa,https://github.com/OceannTwT/SPA,a8bb190a182c076f80d90ac59921abd1df29b0ae,"Train the SPA model on alpaca_data_en_52k (from the repo), startin...",{'e2e_task': 'Train the SPA model on alpaca_data_en_52k (from the ...,"{""training_loss"": 4.97342586517334}","[Generating train split: \d+ examples, >> \*\*\*\*\* Running train...","{'thought_0': 'First, I need to check the contents of the current ...",Stepwise Reasoning: 1. **Repository Preparation:** Upon beginning ...,success=False structured_output={'training_loss': 0.0} reasoning='...,"✔️ [Prediction(\n score=0.0,\n score_dict={'submitted': 1, '..."
2,mezo,https://github.com/princeton-nlp/MeZO,552cb1b710767f9a6e1dc8f9645d7640376f9941,"Train using the ""MeZO"" method (no prefix-tuning / lora) on the RTE...","{'e2e_task': 'Train using the ""MeZO"" method (no prefix-tuning / lo...","{""accuracy"": 0.8, ""dev_accuracy"": 0.4}",['- INFO - Sample train set \\d+/\\d+' '- INFO - \\*\\*\\*\\*\\* R...,"{'thought_0': 'Before running the experiment, I need to verify tha...",Stepwise protocol review: Step 1: Repository Preparation - Checked...,"success=False structured_output={'accuracy': 0.0, 'dev_accuracy': ...","✔️ [Prediction(\n score=0.0,\n score_dict={'submitted': 1, '..."
3,mode-connectivity-plm,https://github.com/thunlp/mode-connectivity-plm,f45efd03fabdb23777c3ddcb35612cb1f7b1960c,"Fine-tune roberta-base on a single-task, ""rotten tomatoes"" which i...","{'e2e_task': 'Fine-tune roberta-base on a single-task, ""rotten tom...","{""eval_loss"": 0.0368}","[Running tokenizer on dataset:, \*\*\*\*\* train metrics \*\*\*\*\...","{'thought_0': ""Before proceeding with the experiment setup, I need...",Step-by-step reasoning for this experiment request: 1. **Repositor...,"success=False structured_output={'eval_loss': 0.0} reasoning=""The ...","✔️ [Prediction(\n score=0.0,\n score_dict={'submitted': 1, '..."
4,mbib,https://github.com/Media-Bias-Group/MBIB,b9a887ffd461fa462e89835fc27b36e370091954,"Train a bart-base model on the ""linguistic-bias"" task using the ba...","{'e2e_task': 'Train a bart-base model on the ""linguistic-bias"" tas...","{""average_weighted_f1"": 0.44272727272727275}","[Training Initialized for fold \d+, The current dev loss: tensor\(...",{'thought_0': 'I need to begin by inspecting the current directory...,"Step 1: Repository Verification Upon starting, I inspected the cur...",success=False structured_output={'average_weighted_f1': 0.0} reaso...,"✔️ [Prediction(\n score=0.0,\n score_dict={'submitted': 1, '..."
5,unsupervisedhierarchicalsymbolicregression,https://github.com/SiyuLou/UnsupervisedHierarchicalSymbolicRegression,9fb1ba66ee1006f14c15f1b5b2390c8d0fa24fd6,Train a hierarchical symbolic regression model from this repo usin...,{'e2e_task': 'Train a hierarchical symbolic regression model from ...,"{""Rf_final"": {""R_square"": -0.25985264778137207, ""MSE"": 0.148261207...","[the size of train dataset is \d+, validation_dataset is \d+, test...","{'thought_0': ""To begin, I need to verify that I'm in the correct ...",Step 1: Repository Preparation I began by verifying the presence o...,"success=False structured_output={'Rf_final': {'R_square': 0.0, 'MS...","✔️ [Prediction(\n score=0.08333333333333333,\n

9.1

Available Tools for team: on runtime aicodetools-team-0f6d0b45 4
Available Tools for cet: on runtime aicodetools-cet-eddc9a60 4
Available Tools for linkbert: on runtime aicodetools-linkbert-b1b54d9b 4
Cleaned up Tools team : True  True
success=False structured_output={'classification_acc': 0.0, 'classification_macro_f1': 0.0, 'instance_acc': 0.0} reasoning='The TEAM codebase at commit e43753fde8e53e498cf3056b85ae2f306902121f was not found in the working directory (/workspace) or its subdirectories. There are no scripts, folders, or files to run, inspect, or modify per task requirements. Therefore, the experiment protocol was halted before any dataset preparation, script inspection, or experiment configuration could occur. Returned placeholder metrics as required.' summary='Terminated at repository preparation step; the TEAM codebase was entirely missing, so no experiment setup or execution could occur. Placeholder metrics provided, success set to false.'
Step 1: Repository Preparation 

GEPA was able to optimize the base program **from 57% score to 61% score** in just 9 iterations. With higher budget, the optimized program's score can go as high as **64%**.

### Let's print the prompts that GEPA discovered

In [ ]:
for name, pred in optimized_program.named_predictors():
    print("================================")
    print(f"Predictor: {name}")
    print("================================")
    print("Prompt:")
    print(pred.signature.instructions)
    print("*********************************")